# Lesson 5: Query Planning and Long-Horizon Agentic RAG

这一节承接 `Lesson 3` 和 `Lesson 4`，专门补上两个在原课程里还没展开的关键能力：

1. `query planning`
2. `long-horizon tool planning`

在前面的 notebook 里，你已经看到：

- 检索可以被包装成工具
- agent 可以决定调用哪个工具
- agent 可以多步推理
- 多文档 agent 可以组织起来

但还有两个更深的问题没有展开：

- 面对复杂问题，agent 该怎么先做“查询计划”？
- 当任务跨多个论文、多个工具、多轮证据整合时，agent 该怎么维持一个更长的执行路径？

这节就专门回答这两个问题。


## 这一节的技术在做什么

这一节讲的是：

- `query planning`
- `long-horizon tool planning`

它们都属于更深入的 `Agentic RAG` 能力。

在前面的课程里，你已经看到：

- 文档检索可以被包装成工具
- agent 可以选择调用哪个工具
- agent 可以跨多篇论文做问答

但那时仍然有一个默认前提：

- agent 虽然会调工具，但“整体检索路径”还是比较隐式

换句话说，你能看到 agent 在做事，但不一定清楚它为什么先查这个、再查那个。

`query planning` 要解决的问题就是：

- 面对复杂问题时，能不能先显式拆成若干检索步骤？
- 每一步的目标是什么？
- 为什么这一步该调用这个工具？

这会让 Agentic RAG 从“会用工具”进一步升级成“会规划证据采集路径”。

`long-horizon tool planning` 进一步解决的是：

- 如果任务不是 1 次调用就能完成，而是 3 步、4 步甚至更多步怎么办？
- 如果问题需要跨多个文档、多个视角、多个子问题，agent 如何维持一个更长的执行链？

你可以把这一节理解成：

- `Lesson 2` 教你“tool calling”
- `Lesson 3` 教你“reasoning loop”
- `Lesson 4` 教你“multi-document agent”
- `Lesson 5` 教你“让整个检索与工具使用过程变得可规划、可解释、可延长”

这一节的核心收获不是某个 API，而是一个模式：

1. 先规划
2. 再执行
3. 再整合证据
4. 最后输出答案

这类模式在真实复杂 Agentic RAG 系统里非常常见，因为很多复杂问题都不适合“想到什么就查什么”。


## Setup


In [ ]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Any, Optional

import nest_asyncio
from pydantic import BaseModel, Field

nest_asyncio.apply()


In [ ]:
# 这一组 notebook 不是重新造一套数据，而是直接复用 Lesson 4 里的论文和工具工厂。
# 这样你学到的是“在原有 Agentic RAG 系统上继续加高级模式”。

lesson4_dir = (Path.cwd() / "../Lesson_4").resolve()
if str(lesson4_dir) not in sys.path:
    sys.path.append(str(lesson4_dir))

from helper import get_dashscope_api_key
from utils import get_doc_tools

from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai_like import OpenAILike


def find_workspace_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if candidate.name == "AIAgent":
            return candidate
    raise FileNotFoundError("Could not find the AIAgent workspace root.")


def find_local_bge_snapshot() -> Path:
    workspace_root = find_workspace_root()
    snapshot_root = workspace_root / "models" / "models--BAAI--bge-small-en-v1.5" / "snapshots"
    snapshots = sorted(
        path for path in snapshot_root.iterdir()
        if path.is_dir() and (path / "config.json").exists()
    )
    if not snapshots:
        raise FileNotFoundError(f"No valid local BGE snapshot found under {snapshot_root}")
    return snapshots[0]


# 继续沿用课程中的 qwen-max + 本地 BGE embedding 配置。
llm = OpenAILike(
    api_key=get_dashscope_api_key(),
    api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen-max",
    temperature=0.1,
    context_window=128000,
    is_chat_model=True,
    is_function_calling_model=True,
)
Settings.llm = llm
Settings.embed_model = HuggingFaceEmbedding(
    model_name=str(find_local_bge_snapshot()),
    device="cpu",
)


papers = [
    "metagpt.pdf",
    "longlora.pdf",
    "selfrag.pdf",
]


def build_tool_registry() -> dict[str, Any]:
    registry = {}
    for paper in papers:
        paper_name = Path(paper).stem
        paper_path = lesson4_dir / paper
        vector_tool, summary_tool = get_doc_tools(str(paper_path), paper_name)
        registry[vector_tool.metadata.name] = vector_tool
        registry[summary_tool.metadata.name] = summary_tool
    return registry


tool_registry = build_tool_registry()
tool_names = list(tool_registry.keys())
print("Loaded tools:", tool_names)


def extract_json_object(text: str) -> dict[str, Any]:
    # qwen-max 有时会返回 ```json 代码块，有时直接返回 JSON。
    # 这里统一把最外层 JSON 对象提出来，再交给 Pydantic 做结构校验。
    cleaned = text.strip()
    cleaned = cleaned.replace("```json", "```").replace("```JSON", "```")
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`").strip()
    match = re.search(r"\{.*\}", cleaned, re.S)
    if not match:
        raise ValueError(f"Could not find JSON object in model output:\n{cleaned}")
    return json.loads(match.group(0))


def complete_json(prompt: str) -> dict[str, Any]:
    response = llm.complete(prompt)
    return extract_json_object(response.text)


def call_tool(tool_name: str, query: str, page_numbers: Optional[list[str]] = None) -> dict[str, Any]:
    # 课程里的工具分成两类：
    # 1. vector_tool_* 需要 query，可选 page_numbers
    # 2. summary_tool_* 通常只需要一个输入问题
    tool = tool_registry[tool_name]
    page_numbers = page_numbers or []

    attempts: list[dict[str, Any]]
    if tool_name.startswith("vector_tool_"):
        attempts = [
            {"query": query, "page_numbers": page_numbers},
            {"query": query},
            {"input": query},
        ]
    else:
        attempts = [
            {"input": query},
            {"query": query},
        ]

    last_error = None
    for kwargs in attempts:
        try:
            tool_output = tool.call(**kwargs)
            raw_output = getattr(tool_output, "raw_output", None)
            source_nodes = getattr(raw_output, "source_nodes", []) if raw_output is not None else []
            source_pages = [
                node.metadata.get("page_label")
                for node in source_nodes
                if hasattr(node, "metadata")
            ]
            return {
                "tool_name": tool_name,
                "query": query,
                "content": str(getattr(tool_output, "content", tool_output)),
                "source_pages": [p for p in source_pages if p],
            }
        except Exception as exc:  # noqa: BLE001
            last_error = exc

    raise RuntimeError(f"Tool call failed for {tool_name}: {last_error}")


def build_tool_catalog() -> str:
    # 这份目录会喂给 planner / critic / evaluator。
    # 它相当于让模型先知道“我手上有哪些工具，每个工具大概适合干什么”。
    lines = []
    for tool_name, tool in tool_registry.items():
        lines.append(f"- {tool_name}: {tool.metadata.description}")
    return "\n".join(lines)


tool_catalog = build_tool_catalog()


## 1. Define the planning schema

这里我们不直接让 agent 黑盒式调用工具，而是先显式让模型产出一个“计划对象”。

这样做的学习价值很高，因为你能清楚看到：

- 计划分成了哪些步骤
- 每一步为什么选这个工具
- 哪些步骤是顺序依赖的


In [ ]:
class PlanStep(BaseModel):
    step_id: int = Field(description="The execution order of this step.")
    objective: str = Field(description="What this step is trying to learn.")
    tool_name: str = Field(description="Which tool should be called.")
    query: str = Field(description="The exact query to send to the tool.")
    why_this_tool: str = Field(description="Why this tool is appropriate for the step.")


class QueryPlan(BaseModel):
    question: str = Field(description="The original user question.")
    reasoning: str = Field(description="High-level reasoning for the overall plan.")
    steps: list[PlanStep] = Field(description="Ordered plan steps.")
    success_criteria: list[str] = Field(description="What evidence would count as a successful run.")


In [ ]:
def create_query_plan(user_question: str) -> QueryPlan:
    # 这里的 planner 不直接回答问题，而是先做“任务拆解”。
    # 这正是 query planning 的关键：面对复杂问题，先规划证据采集路径。
    prompt = f'''
    You are planning a multi-step retrieval workflow over research-paper tools.

    Available tools:
    {tool_catalog}

    User question:
    {user_question}

    Return a JSON object with the following shape:
    {{
      "question": "...",
      "reasoning": "...",
      "steps": [
        {{
          "step_id": 1,
          "objective": "...",
          "tool_name": "...",
          "query": "...",
          "why_this_tool": "..."
        }}
      ],
      "success_criteria": ["...", "..."]
    }}

    Rules:
    - Use only the available tools.
    - Prefer summary tools for broad orientation and vector tools for targeted evidence.
    - If the question compares papers, plan for evidence collection from each relevant paper.
    - Keep the plan to 2-5 steps.
    - Return JSON only.
    '''
    plan_dict = complete_json(prompt)
    return QueryPlan.model_validate(plan_dict)


## 2. Execute the plan step by step

这一步开始体现 `long-horizon tool planning`：

- 不再是“问一句，调一个工具”
- 而是先拿到一个多步计划
- 再按步骤执行
- 最后把所有证据汇总给模型做综合回答


In [ ]:
def execute_query_plan(plan: QueryPlan) -> list[dict[str, Any]]:
    execution_trace = []

    for step in sorted(plan.steps, key=lambda item: item.step_id):
        # 每个 step 都是一条显式的检索行动。
        result = call_tool(step.tool_name, step.query)
        execution_trace.append(
            {
                "step_id": step.step_id,
                "objective": step.objective,
                "tool_name": step.tool_name,
                "query": step.query,
                "why_this_tool": step.why_this_tool,
                "evidence": result["content"],
                "source_pages": result["source_pages"],
            }
        )

    return execution_trace


def synthesize_final_answer(question: str, plan: QueryPlan, execution_trace: list[dict[str, Any]]) -> str:
    # planner 负责“怎么查”，这里的 synthesizer 负责“怎么答”。
    # 这也是 Agentic RAG 的常见拆分方式：规划层、执行层、综合层分开。
    evidence_blocks = []
    for item in execution_trace:
        evidence_blocks.append(
            f"Step {item['step_id']}\n"
            f"Objective: {item['objective']}\n"
            f"Tool: {item['tool_name']}\n"
            f"Query: {item['query']}\n"
            f"Source pages: {item['source_pages']}\n"
            f"Evidence:\n{item['evidence']}"
        )

    evidence_text = "\n\n".join(evidence_blocks)

    prompt = f'''
    You are synthesizing a final answer from a multi-step retrieval workflow.

    Original user question:
    {question}

    Planner reasoning:
    {plan.reasoning}

    Success criteria:
    {plan.success_criteria}

    Retrieved evidence:
    {evidence_text}

    Write a grounded answer that only uses the evidence above.
    Also mention where the evidence appears weak or incomplete.
    '''
    return llm.complete(prompt).text


In [ ]:
# 这个问题是对 Lesson 4 问题形式的自然升级：
# 不是简单问一篇论文，而是要求跨论文对比并判断哪篇论文真正处理了“检索”这个问题。
complex_question = (
    "Compare how MetaGPT organizes agent collaboration with how Self-RAG controls retrieval, "
    "and explain whether LongLoRA changes retrieval behavior or mainly improves long-context handling."
)

plan = create_query_plan(complex_question)
plan


In [ ]:
execution_trace = execute_query_plan(plan)

for item in execution_trace:
    print(f"Step {item['step_id']}: {item['objective']}")
    print(f"Tool: {item['tool_name']}")
    print(f"Query: {item['query']}")
    print(f"Source pages: {item['source_pages']}")
    print(item["evidence"][:1000])
    print("-" * 100)


In [ ]:
final_answer = synthesize_final_answer(complex_question, plan, execution_trace)
print(final_answer)
